# TAR conformational space


This notebook contains the analysis workflow used to compare TAR RNA conformational states, cluster the sampled ensemble, and relate representative conformations to available bound structures.


In [ ]:
import os
import math
import pickle
import tempfile
import copy
from pathlib import Path

import MDAnalysis as mda
import barnaba as bb
import bussilab
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from bussilab import clustering
from matplotlib.colors import Colormap, LinearSegmentedColormap, Normalize
from matplotlib.legend_handler import HandlerPathCollection
from MDAnalysis.lib.distances import distance_array
from scipy.ndimage import gaussian_filter
from scipy.spatial.distance import cdist
from sklearn.cluster import DBSCAN, HDBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import pairwise_distances



In [ ]:
traj="data/rna_repl_00.xtc"
top="data/rna.gro"

trj=mda.Universe(top,traj)

gvecs=bb.dump_gvec(traj,top)

## Pockets


Pocket residues are identified from the available ligand-bound structures using a distance cutoff, then the same residue sets are projected onto the simulation trajectory.


In [ ]:
PDB_DIR = "structures"
CUTOFF_ANGSTROM = 4.5

PDB_CODES = [
    "1ANR", "1ARJ", "1QD3", "1UTS", "1UUD", "1UUI", "2KX5", "2KDQ", "2L8H", "5J0M",
    "5J1O", "5J2W", "6D2U", "7JU1", "8THV", "1LVJ", "6XH1", "6XH2", "6XH3"
]

STANDARD_AA = {
    "ALA", "ARG", "ASN", "ASP", "CYS", "GLN", "GLU", "GLY", "HIS", "ILE",
    "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"
}
STANDARD_NT = {
    "A", "U", "G", "C", "I", "DA", "DT", "DG", "DC", "DI"
}

In [ ]:
def analyse_structures():
    pockets = {}

    for code in PDB_CODES:
        pdb_path = os.path.join(PDB_DIR, f"{code}.pdb")
        if not os.path.isfile(pdb_path):
            continue

        u = mda.Universe(pdb_path)

        polymer_atoms = u.select_atoms("protein or nucleic")
        ligand_atoms = u.select_atoms("not (protein or nucleic or resname HOH)")

        if ligand_atoms.n_atoms == 0:
            continue
        pocket_set = set()
        for lig_res in ligand_atoms.residues:
            dist = distance_array(
                lig_res.atoms.positions,
                polymer_atoms.positions,
                box=u.dimensions,
            )
            close_mask = (dist <= CUTOFF_ANGSTROM).any(axis=0)
            for res in polymer_atoms[close_mask].residues:
                resname = res.resname.strip()
                pocket_set.add(f"{resname}-{res.resid}")

        if pocket_set:
            pockets[code] = sorted(
                pocket_set,
                key=lambda s: (s.split("-")[0], int(s.split("-")[1]))
            )

    return pockets

In [ ]:
pockets = analyse_structures()

In [ ]:
EXCLUDE_GVEC = {"2KX5", "2KDQ", "5J0M", "5J1O", "5J2W", "6D2U"}

def compute_pocket_gvecs(pocket_dict, out_dir="gvec_pockets"):
    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    gvecs = {}
    for code, residues in pocket_dict.items():
        if code.upper() in EXCLUDE_GVEC:
            continue

        pdb_path = Path(PDB_DIR) / f"{code}.pdb"
        u = mda.Universe(str(pdb_path))

        resids = [int(r.split("-")[1]) for r in residues]
        sel = " or ".join(f"resid {rid}" for rid in resids) + " and nucleic"
        pocket_atoms = u.select_atoms(sel)
        if pocket_atoms.n_atoms == 0:
            continue

        with tempfile.NamedTemporaryFile(suffix=".pdb", delete=False) as tmp:
            with mda.Writer(tmp.name) as w:
                w.write(pocket_atoms)
            tmp_pdb = tmp.name

        try:
            gvec = bb.dump_gvec(tmp_pdb)
            gvecs[code] = gvec
        finally:
            os.unlink(tmp_pdb)

    return gvecs

In [ ]:
gvecs = compute_pocket_gvecs(pockets)

In [ ]:
def compute_dynamic_pocket_gvecs(pocket_dict, traj_xtc, top_gro, out_dir="gvec_pockets_dynamic"):

    out_dir = Path(out_dir)
    out_dir.mkdir(exist_ok=True)

    u_traj = mda.Universe(str(top_gro), str(traj_xtc))

    gvecs_dyn = {}
    for code, residues in pocket_dict.items():
        if code.upper() in EXCLUDE_GVEC:
            continue
        resids = [int(r.split("-")[1]) - 16 for r in residues]
        sel = f"nucleic and resid {' '.join(map(str, resids))}"
        try:
            pocket_atoms = u_traj.select_atoms(sel)
        except Exception:
            continue

        if pocket_atoms.n_atoms == 0:
            continue

        with tempfile.TemporaryDirectory() as tmpdir:
            tmp_gro = Path(tmpdir) / f"{code}_pocket.gro"
            tmp_xtc = Path(tmpdir) / f"{code}_pocket.xtc"

            with mda.Writer(str(tmp_gro)) as wgro:
                wgro.write(pocket_atoms)
            with mda.Writer(str(tmp_xtc), n_atoms=pocket_atoms.n_atoms) as wxtc:
                for ts in u_traj.trajectory:
                    wxtc.write(pocket_atoms)
            try:
                gvec = bb.dump_gvec(str(tmp_xtc), str(tmp_gro))
                gvecs_dyn[code] = gvec
            except Exception:
                continue

    return gvecs_dyn

In [ ]:
gvecs_dyn = compute_dynamic_pocket_gvecs(pockets, traj, top)

## Global PCA


The global projection combines experimental structures and trajectory frames in a common g-vector space. This representation is used to visualize the sampled ensemble and the effect of reweighting.


In [ ]:
weights_data=pickle.load(open("data/weights_sigmas.pkl", "rb"))

In [ ]:
w_full=weights_data[7]

In [ ]:


gvecs_1anr=bb.dump_gvec("structures/1ANR.pdb")
gvecs_1arj=bb.dump_gvec("structures/1ARJ.pdb")
gvecs_1qd3=bb.dump_gvec("structures/1QD3.pdb")
gvecs_1uts=bb.dump_gvec("structures/1UTS.pdb")
gvecs_1uud=bb.dump_gvec("structures/1UUD.pdb")
gvecs_1uui=bb.dump_gvec("structures/1UUI.pdb")
gvecs_2kx5=bb.dump_gvec("structures/2KX5.pdb")
gvecs_2kdq=bb.dump_gvec("structures/2KDQ.pdb")
gves_2l8h=bb.dump_gvec("structures/2L8H.pdb")
gves_5J0M=bb.dump_gvec("structures/5J0M.pdb")
gves_5J1O=bb.dump_gvec("structures/5J1O.pdb")
gvecs_5j2w=bb.dump_gvec("structures/5J2W.pdb")
gvecs_6d2u=bb.dump_gvec("structures/6D2U.pdb")
gvecs_7ju1=bb.dump_gvec("structures/7JU1.pdb")
gvecs_8thv=bb.dump_gvec("structures/8THV.pdb")
gvecs_1lvj=bb.dump_gvec("structures/1LVJ.pdb")
gvecs_6xh1=bb.dump_gvec("structures/6XH1.pdb")
gvecs_6xh2=bb.dump_gvec("structures/6XH2.pdb")
gvecs_6xh3=bb.dump_gvec("structures/6XH3.pdb")
gvecs_comm_1lvj=gvecs_1lvj[0][:,4:27,4:27]

In [ ]:
gvecs_comm_1anr=gvecs_1anr[0][:,3:26,3:26]
gvecs_comm_1arj=gvecs_1arj[0][:,3:26,3:26]
gvecs_comm_1qd3=gvecs_1qd3[0][:,3:26,3:26]
gvecs_comm_1uts=gvecs_1uts[0][:,3:26,3:26]
gvecs_comm_1uud=gvecs_1uud[0][:,3:26,3:26]
gvecs_comm_1uui=gvecs_1uui[0][:,3:26,3:26]
gvecs_comm_2kx5=gvecs_2kx5[0][:,3:26,3:26]
gvecs_comm_2kdq=gvecs_2kdq[0][:,3:26,3:26]
gvecs_comm_2l8h=gves_2l8h[0][:,3:26,3:26]
gvecs_comm_5j0m=gves_5J0M[0][:,3:26,3:26]
gvecs_comm_5j1o=gves_5J1O[0][:,3:26,3:26]
gvecs_comm_5j2w=gvecs_5j2w[0][:,3:26,3:26]
gvecs_comm_6d2u=gvecs_6d2u[0][:,3:26,3:26]
gvecs_comm_7ju1=gvecs_7ju1[0][:,3:26,3:26]
gvecs_comm_8thv=gvecs_8thv[0][:,3:26,3:26]
gvecs_comm_6xh1=gvecs_6xh1[0][:,2:25,2:25]
gvecs_comm_6xh2=gvecs_6xh2[0][:,2:25,2:25]
gvecs_comm_6xh3=gvecs_6xh3[0][:,2:25,2:25]


In [ ]:

concat_trj_structures=np.concatenate((gvecs_comm_1lvj,gvecs_comm_1anr,gvecs_comm_1arj,gvecs_comm_1qd3,gvecs_comm_1uts,gvecs_comm_1uud,gvecs_comm_1uui,gvecs_comm_2kx5,gvecs_comm_2kdq,gvecs_comm_2l8h,gvecs_comm_5j0m,gvecs_comm_5j1o,gvecs_comm_5j2w,gvecs_comm_6d2u,gvecs_comm_7ju1,gvecs_comm_8thv,gvecs_comm_6xh1,gvecs_comm_6xh2,gvecs_comm_6xh3,gvecs[0][:,3:26,3:26]),axis=0)



In [ ]:
pca=PCA(n_components=2)
dim_red_all=pca.fit_transform(concat_trj_structures.reshape(70201,-1))



In [ ]:
palette = sns.color_palette("tab20", 20)
kBT=2.5
w = np.maximum(weights_data[7], 1e-12)
deltaG = -kBT * np.log(w)
sizes = 1000 * np.exp(-0.15 * deltaG)

In [ ]:
X = dim_red_all[200:70201]
w = weights_data[7][:len(X)] * len(X)

db = DBSCAN(eps=0.08, min_samples=40, metric="euclidean")
labels = db.fit_predict(X, sample_weight=w)

In [ ]:
idx_unbound = np.r_[0:20, 145:165, 165:185]
idx_bound   = np.r_[20:40, 185:197, 40:57, 57:73, 74, 75, 95:105, 105:115,
                    76:85, 85:95, 115:125, 125:135, 135:145, 197, 198, 199]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
sns.scatterplot(
    x=dim_red_all[idx_unbound, 0], y=dim_red_all[idx_unbound, 1], ax=axes[0],
    s=40, marker="x", linewidths=1.4, color="#ff595e",
    alpha=0.95, zorder=3, label="Unbound states", legend=False
)
sns.scatterplot(
    x=dim_red_all[idx_bound, 0], y=dim_red_all[idx_bound, 1], ax=axes[0],
    s=40, marker="x", linewidths=1.4, color="#1982c4",
    alpha=0.95, zorder=3, label="Bound states", legend=False
)
sns.scatterplot(
    x=dim_red_all[200:70201, 0], y=dim_red_all[200:, 1], ax=axes[0],
    s=10, color="lightgrey", alpha=0.8, zorder=1, edgecolor="grey", linewidth=0.3,
    rasterized=True, label="MD trajectory frames", legend=False
)
axes[0].set_xlabel("PC1")
axes[0].set_ylabel("PC2")

X = dim_red_all[200:70201, :2]

sns.scatterplot(
    x=dim_red_all[200:70201, 0], y=dim_red_all[200:, 1], ax=axes[1],
    s=sizes, color="lightgrey", alpha=1.0, zorder=1, edgecolor="grey", linewidth=0.2,
    rasterized=True
)

sns.kdeplot(
    x=dim_red_all[200:70201, 0], y=dim_red_all[200:, 1], ax=axes[1],
    fill=False, color="grey", levels=8, thresh=0.02, bw_adjust=0.65,
    weights=w_full, linewidths=1.2, zorder=4
)
tbplotted=[6,4,19,15,31]

hex_palette = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"]

selected_present = [c for c in tbplotted if np.any(lab == c)]
cluster_color = {c: hex_palette[i] for i, c in enumerate(selected_present)}

it=1
for i, c in enumerate(selected_present):
    m = (lab == c)
    sns.scatterplot(
        x=X[m, 0], y=X[m, 1], ax=axes[1],
        s=sizes[np.where(lab==c)[0]], color=cluster_color[c], alpha=0.85, linewidth=0, zorder=3,
        label=f"Cluster {it}",legend=False
    )

    iw = centers_w[i]
    axes[1].scatter(
        X[iw, 0], X[iw, 1],
        marker="*", s=280, c=[cluster_color[c]],
        edgecolor="black", linewidth=1.0, zorder=5
    )
    it+=1
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("PC2")
axes[1].sharey(axes[0])

h0, l0 = axes[0].get_legend_handles_labels()
h1, l1 = axes[1].get_legend_handles_labels()
handles, labels = [], []
for h, l in list(zip(h0, l0)) + list(zip(h1, l1)):
    if l not in labels:
        handles.append(h); labels.append(l)

class FixedSizeHandler(HandlerPathCollection):
    def __init__(self, size=100, **kwargs):
        super().__init__(**kwargs)
        self._size = size
    def update_prop(self, legend_handle, orig_handle, legend):
        super().update_prop(legend_handle, orig_handle, legend)
        legend_handle.set_sizes([self._size])

fig.tight_layout(rect=[0, 0.12, 1, 1])
fig.legend(handles, labels,
           loc="lower center", bbox_to_anchor=(0.5, 0.00),
           frameon=False, fontsize=15, ncol=4,
           handler_map={type(handles[0]): FixedSizeHandler(100),
                        type(handles[1]): FixedSizeHandler(100),
                        type(handles[2]): FixedSizeHandler(100)})

plt.savefig("./images_paper/conf_space_before_after_legends.png", dpi=600, bbox_inches='tight')
plt.savefig("./images_paper/conf_space_before_after_legends.svg", dpi=600, bbox_inches='tight')

## Pocket PCA


For each binding pocket, the corresponding g-vector subset is extracted so local pocket motions can be compared independently from the full RNA projection.


In [ ]:
gvecs_1anr=bb.dump_gvec("structures/1ANR.pdb")
gvecs_1arj=bb.dump_gvec("structures/1ARJ.pdb")
gvecs_1qd3=bb.dump_gvec("structures/1QD3.pdb")
gvecs_1uts=bb.dump_gvec("structures/1UTS.pdb")
gvecs_1uud=bb.dump_gvec("structures/1UUD.pdb")
gvecs_1uui=bb.dump_gvec("structures/1UUI.pdb")
gvecs_2kx5=bb.dump_gvec("structures/2KX5.pdb")
gvecs_2kdq=bb.dump_gvec("structures/2KDQ.pdb")
gvecs_2l8h=bb.dump_gvec("structures/2L8H.pdb")
gvecs_5J0M=bb.dump_gvec("structures/5J0M.pdb")
gvecs_5J1O=bb.dump_gvec("structures/5J1O.pdb")
gvecs_5j2w=bb.dump_gvec("structures/5J2W.pdb")
gvecs_6d2u=bb.dump_gvec("structures/6D2U.pdb")
gvecs_7ju1=bb.dump_gvec("structures/7JU1.pdb")
gvecs_8thv=bb.dump_gvec("structures/8THV.pdb")
gvecs_1lvj=bb.dump_gvec("structures/1LVJ.pdb")
gvecs_6xh1=bb.dump_gvec("structures/6XH1.pdb")
gvecs_6xh2=bb.dump_gvec("structures/6XH2.pdb")
gvecs_6xh3=bb.dump_gvec("structures/6XH3.pdb")

matching_indices = [i for i, x in enumerate(gvecs_1qd3[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1QD3"]]

gvecs_pockets_1qd3 = gvecs_1qd3[0][:, matching_indices, :][:, :, matching_indices]

matching_indices = [i for i, x in enumerate(gvecs_1uts[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1UTS"]]

gvecs_pockets_1uts = gvecs_1uts[0][:, matching_indices, :][:, :, matching_indices]

matching_indices = [i for i, x in enumerate(gvecs_1uts[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1UTS"]]

gvecs_pockets_1uts = gvecs_1uts[0][:, matching_indices, :][:, :, matching_indices]

In [ ]:
results = {}

for pdb in pockets.keys():
    if pdb in EXCLUDE_GVEC:
        pass
    else:
        var_name = f"gvecs_{pdb.lower()}"
        gvecs_data = globals()[var_name]

        matching_indices = [i for i, x in enumerate(gvecs_data[1])
                        if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets[pdb]]

        gvecs_pockets = gvecs_data[0][:, matching_indices, :][:, :, matching_indices]
        results[pdb] = gvecs_pockets

## Global ensemble clustering


The trajectory is clustered in g-vector space. A subsampled distance matrix is used first, and skipped frames are assigned back to the closest representative clusters.


### Whole trajectory


In [ ]:
dist_mat = pairwise_distances(X = gvecs[0][:,3:26,3:26][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat = dist_mat / np.sqrt(gvecs[0][:,3:26,3:26].shape[1])

subset_idx   = np.arange(len(gvecs[0][:,3:26,3:26]))[::2]
excluded_idx = np.setdiff1d(np.arange(len(gvecs[0][:,3:26,3:26])), subset_idx)

cluster_ensemble_8=clustering.qt(dist_mat,0.8)

centroids={}
for i in range(len(cluster_ensemble_8["clusters"])):
    dist_intra=dist_mat[cluster_ensemble_8["clusters"][i],:][:,cluster_ensemble_8["clusters"][i]]
    centroids[i]=cluster_ensemble_8["clusters"][i][np.argmin(np.sum(dist_intra,axis=0))]

centroid_indices = list(centroids.values())

centroid_distances = dist_mat[np.ix_(centroid_indices, centroid_indices)]

plt.matshow(centroid_distances)
plt.colorbar()

dist_mat = pairwise_distances(X = gvecs[0][:,3:26,3:26][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat = dist_mat / np.sqrt(gvecs[0][:,3:26,3:26][::2].shape[1])

dist_centroids=dist_mat[np.ix_(centroid_indices, excluded_idx)]

### Cluster refinement


In [ ]:
adjusted_clusters = copy.deepcopy(cluster_ensemble_8)

dist_centroids_initial= dist_mat[np.ix_(list(centroids.values()), list(centroids.values()))]

plt.matshow(dist_centroids_initial)
plt.colorbar()

min_weight_small = 10
min_weight_big   = 100
tau_merge        = 0.6

included_idx=np.arange(len(gvecs[0][:,3:26,3:26]))[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(cluster_ensemble_8["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[cluster_ensemble_8["clusters"][i]])
    cluster_correct_idx["weights"].append(cluster_ensemble_8["weights"][i])

### Add skipped frames


In [ ]:
centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

In [ ]:
weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

weights = sorted_cluster_dict["weights"]
plt.style.use('default')
plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70001, color="#1f77b4", alpha=0.9)

plt.yticks(fontsize=15)
plt.xticks(fontsize=15)
plt.xlabel("cluster index", fontsize=15)
plt.tight_layout()
plt.show()

In [ ]:
weights = sorted_cluster_dict["weights"]


In [ ]:
weights = sorted_cluster_dict["weights"][:200]

with open("data/clustering_ensemble_noreweights.pkl", "rb") as f:
    sorted_cluster_dict = pickle.load(f)

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in sorted_cluster_dict["clusters"]
]


In [ ]:
medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

In [ ]:
large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70002)

plt.tight_layout()
plt.show()

with open("data/weights_sigmas.pkl", "rb") as f:
    weights_data = pickle.load(f)

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

with open("data/recap_dictionary_clustering.pkl", "rb") as f:
    recap_dictionary = pickle.load(f)

weights = recap_dictionary["weights"]


In [ ]:
weights = recap_dictionary["reweights"]

## Crystal distances


Cluster medoids are compared with the crystal structures to identify the sampled conformations closest to known ligand-bound states.


In [ ]:
gvecs_medoids = np.zeros((len(medoids), 23,23,4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs[0][int(i),3:26,3:26]

pdb_names=["1anr","1arj","1qd3","1uts","1uud","1uui","2kx5","2kdq","2l8h","5j0m","5j1o","5j2w","6d2u","7ju1","8thv","6xh1","6xh2","6xh3"]

for i in pdb_names:
    gvecs_comm = globals()[f'gvecs_comm_{i}']
    dist = cdist(gvecs_medoids.reshape(len(medoids), -1), gvecs_comm.reshape(len(gvecs_comm),-1), metric='euclidean')
    dist = dist / np.sqrt(gvecs_medoids.shape[1])
    closest_centroid = int(np.where(dist==np.min(dist))[1])

## Pocket clustering


The same clustering and reweighting analysis is repeated on individual pockets to quantify local similarity to bound pocket geometries.


In [ ]:
gvecs_1anr=bb.dump_gvec("structures/1ANR.pdb")
gvecs_1arj=bb.dump_gvec("structures/1ARJ.pdb")
gvecs_1qd3=bb.dump_gvec("structures/1QD3.pdb")
gvecs_1uts=bb.dump_gvec("structures/1UTS.pdb")
gvecs_1uud=bb.dump_gvec("structures/1UUD.pdb")
gvecs_1uui=bb.dump_gvec("structures/1UUI.pdb")
gvecs_2kx5=bb.dump_gvec("structures/2KX5.pdb")
gvecs_2kdq=bb.dump_gvec("structures/2KDQ.pdb")
gvecs_2l8h=bb.dump_gvec("structures/2L8H.pdb")
gvecs_5J0M=bb.dump_gvec("structures/5J0M.pdb")
gvecs_5J1O=bb.dump_gvec("structures/5J1O.pdb")
gvecs_5j2w=bb.dump_gvec("structures/5J2W.pdb")
gvecs_6d2u=bb.dump_gvec("structures/6D2U.pdb")
gvecs_7ju1=bb.dump_gvec("structures/7JU1.pdb")
gvecs_8thv=bb.dump_gvec("structures/8THV.pdb")
gvecs_1lvj=bb.dump_gvec("structures/1LVJ.pdb")
gvecs_6xh1=bb.dump_gvec("structures/6XH1.pdb")
gvecs_6xh2=bb.dump_gvec("structures/6XH2.pdb")
gvecs_6xh3=bb.dump_gvec("structures/6XH3.pdb")

with open("gvecs_dyn.pkl", "rb") as f:
    gvecs_dyn = pickle.load(f)

### 1QD3 pocket


In [ ]:
dist_mat_1qd3 = pairwise_distances(X =gvecs_dyn["1QD3"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1qd3 = dist_mat_1qd3 / np.sqrt(gvecs_dyn["1QD3"][0][::2].shape[1])

clustering_1qd3 = clustering.qt(dist_mat_1qd3, 0.8)

dist_mat_1qd3 = pairwise_distances(X =gvecs_dyn["1QD3"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1qd3 = dist_mat_1qd3 / np.sqrt(gvecs_dyn["1QD3"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_1qd3["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_1qd3["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_1qd3["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_1qd3[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_1qd3[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

In [ ]:
for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

with open("data/weights_sigmas.pkl", "rb") as f:
    weights_data = pickle.load(f)

In [ ]:
medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_1qd3[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_1qd3[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

with open("data/weights_sigmas.pkl", "rb") as f:
    weights_data = pickle.load(f)

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singletwhiteon["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_1qd3[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

with open("data/recap_dictionary_clustering_1qd3.pkl", "rb") as f:
    recap_dictionary_1qd3 = pickle.load(f)

### Crystal-pocket distance


In [ ]:
matching_indices = [i for i, x in enumerate(gvecs_1qd3[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1QD3"]]

gvecs_pockets_1qd3 = gvecs_1qd3[0][:, matching_indices, :][:, :, matching_indices]

matching_indices = [i for i, x in enumerate(gvecs_1uts[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1UTS"]]

gvecs_pockets_1uts = gvecs_1uts[0][:, matching_indices, :][:, :, matching_indices]

matching_indices = [i for i, x in enumerate(gvecs_1uts[1]) if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets["1UTS"]]

gvecs_pockets_1uts = gvecs_1uts[0][:, matching_indices, :][:, :, matching_indices]

In [ ]:
results = {}

for pdb in pockets.keys():
    if pdb in EXCLUDE_GVEC:
        pass
    else:
        var_name = f"gvecs_{pdb.lower()}"
        gvecs_data = globals()[var_name]

        matching_indices = [i for i, x in enumerate(gvecs_data[1])
                        if f"{x.split('_')[0]}-{x.split('_')[1]}" in pockets[pdb]]

        gvecs_pockets = gvecs_data[0][:, matching_indices, :][:, :, matching_indices]
        results[pdb] = gvecs_pockets

In [ ]:
gvecs_medoids = np.zeros((len(medoids), 11,11,4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["1QD3"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["1QD3"].reshape(len(results["1QD3"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 1UTS pocket


In [ ]:
dist_mat_1uts = pairwise_distances(X =gvecs_dyn["1UTS"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uts = dist_mat_1uts / np.sqrt(gvecs_dyn["1UTS"][0][::2].shape[1])

clustering_1uts  = clustering.qt(dist_mat_1uts, 0.8)

dist_mat_1uts = pairwise_distances(X =gvecs_dyn["1UTS"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uts = dist_mat_1uts / np.sqrt(gvecs_dyn["1UTS"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_1uts["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_1uts["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_1uts["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_1uts[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_1uts[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

In [ ]:
for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"]


weights = sorted_cluster_dict["weights"][:200]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70002)

plt.tight_layout()
plt.show()

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_1uts[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_1uts[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]


In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_1uts[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), 5,5,4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["1UTS"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["1UTS"].reshape(len(results["1UTS"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 1UUD pocket


In [ ]:
dist_mat_1uud = pairwise_distances(X =gvecs_dyn["1UUD"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uud = dist_mat_1uud / np.sqrt(gvecs_dyn["1UUD"][0][::2].shape[1])

clustering_1uud  = clustering.qt(dist_mat_1uud, 0.8)

dist_mat_1uud = pairwise_distances(X =gvecs_dyn["1UUD"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uud = dist_mat_1uud / np.sqrt(gvecs_dyn["1UUD"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_1uud["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_1uud["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_1uud["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_1uud[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_1uud[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

In [ ]:
for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"][:200]

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_1uud[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_1uud[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]
w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]


In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_1uud[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), gvecs_dyn["1UUD"][0].shape[1],gvecs_dyn["1UUD"][0].shape[2],4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["1UUD"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["1UUD"].reshape(len(results["1UUD"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 1UUI pocket


In [ ]:
dist_mat_1uui = pairwise_distances(X =gvecs_dyn["1UUI"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uui = dist_mat_1uui / np.sqrt(gvecs_dyn["1UUI"][0][::2].shape[1])

clustering_1uui  = clustering.qt(dist_mat_1uui, 0.8)

dist_mat_1uui = pairwise_distances(X =gvecs_dyn["1UUI"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1uui = dist_mat_1uui / np.sqrt(gvecs_dyn["1UUI"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_1uui["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_1uui["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_1uui["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_1uui[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_1uui[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

In [ ]:
for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"][:200]

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_1uui[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_1uui[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70001)

plt.tight_layout()
plt.show()

weights_data= pickle.load(open("data/weights_sigmas.pkl", "rb"))

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_1uui[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), gvecs_dyn["1UUI"][0].shape[1], gvecs_dyn["1UUI"][0].shape[2], 4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["1UUI"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["1UUI"].reshape(len(results["1UUI"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 2L8H pocket


In [ ]:
dist_mat_2l8h = pairwise_distances(X =gvecs_dyn["2L8H"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_2l8h = dist_mat_2l8h / np.sqrt(gvecs_dyn["2L8H"][0][::2].shape[1])

del dist_mat_1uui

clustering_2l8h  = clustering.qt(dist_mat_2l8h, 0.8)

dist_mat_2l8h = pairwise_distances(X =gvecs_dyn["2L8H"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_2l8h = dist_mat_2l8h / np.sqrt(gvecs_dyn["2L8H"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_2l8h["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_2l8h["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_2l8h["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_2l8h[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_2l8h[np.ix_(excluded_idx,centroid_indices)]

In [ ]:
membership_matrix = np.zeros(( len(excluded_idx),2))

for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), weights)

plt.yticks([])
plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

weights = sorted_cluster_dict["weights"][:200]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70002)

plt.tight_layout()
plt.show()

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_2l8h[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_2l8h[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70001)

plt.tight_layout()
plt.show()

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_2l8h[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), gvecs_dyn["2L8H"][0].shape[1], gvecs_dyn["2L8H"][0].shape[2], 4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["2L8H"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["2L8H"].reshape(len(results["2L8H"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 2L8H corrected pocket


In [ ]:
dist_mat_2l8h = pairwise_distances(X =gvecs_dyn["2L8H"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_2l8h = dist_mat_2l8h / np.sqrt(gvecs_dyn["2L8H"][0][::2].shape[1])

clustering_2l8h  = clustering.qt(dist_mat_2l8h, 0.8)

with open("data/clustering_2l8h_partial_correct.pkl", "rb") as f:
    clustering_2l8h = pickle.load(f)

del dist_mat_2l8h

dist_mat_2l8h = pairwise_distances(X =gvecs_dyn["2L8H"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_2l8h = dist_mat_2l8h / np.sqrt(gvecs_dyn["2L8H"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_2l8h["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_2l8h["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_2l8h["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_2l8h[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

In [ ]:
excluded_centroid_distances = dist_mat_2l8h[np.ix_(excluded_idx,centroid_indices)]

membership_matrix = np.zeros(( len(excluded_idx),2))

for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), weights)

plt.yticks([])
plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

weights = sorted_cluster_dict["weights"][:200]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70002)

plt.tight_layout()
plt.show()

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_2l8h[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_2l8h[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70001)

plt.tight_layout()
plt.show()

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_2l8h[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), gvecs_dyn["2L8H"][0].shape[1], gvecs_dyn["2L8H"][0].shape[2], 4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["2L8H"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["2L8H"].reshape(len(results["2L8H"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

### 1LVJ pocket


In [ ]:
del dist_mat_2l8h

dist_mat_1lvj = pairwise_distances(X =gvecs_dyn["1LVJ"][0][::2].reshape(35001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1lvj = dist_mat_1lvj / np.sqrt(gvecs_dyn["1LVJ"][0][::2].shape[1])

clustering_1lvj  = clustering.qt(dist_mat_1lvj, 0.8)

dist_mat_1lvj = pairwise_distances(X =gvecs_dyn["1LVJ"][0].reshape(70001, -1), metric = 'euclidean', n_jobs = -1)
dist_mat_1lvj = dist_mat_1lvj / np.sqrt(gvecs_dyn["1LVJ"][0].shape[1])

included_idx=np.arange(70001)[::2]

cluster_correct_idx={}
cluster_correct_idx["clusters"] = []
cluster_correct_idx["weights"] = []
for i in range(len(clustering_1lvj["clusters"])):
    cluster_correct_idx["clusters"].append(included_idx[clustering_1lvj["clusters"][i]])
    cluster_correct_idx["weights"].append(clustering_1lvj["weights"][i])

excluded_idx=np.setdiff1d(np.arange(70001), included_idx)

centroids = {}
for i, cluster in enumerate(cluster_correct_idx["clusters"]):
    if len(cluster) >= 10:
        dist_intra = dist_mat_1lvj[cluster, :][:, cluster]
        centroids[i] = cluster[np.argmin(np.sum(dist_intra, axis=0))]

centroid_indices = list(centroids.values())

excluded_centroid_distances = dist_mat_1lvj[np.ix_(excluded_idx,centroid_indices)]

In [ ]:
membership_matrix = np.zeros(( len(excluded_idx),2))

for i in range(len(excluded_centroid_distances)):
    min_idx = np.argmin(excluded_centroid_distances[i])
    min_dist = excluded_centroid_distances[i, min_idx]
    if min_dist< 0.8:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = min_idx
    else:
        membership_matrix[i, 0] = excluded_idx[i]
        membership_matrix[i, 1] = -1

for i in membership_matrix:
    if i[1] != -1:
        cluster_correct_idx["clusters"][int(i[1])] = np.concatenate((cluster_correct_idx["clusters"][int(i[1])], [i[0]]))
        cluster_correct_idx["weights"][int(i[1])] += 1
    else:
        cluster_correct_idx["clusters"].append(np.array(int(i[0])))
        cluster_correct_idx["weights"].append(1)

weights  = np.asarray(cluster_correct_idx['weights'])
clusters = np.asarray(cluster_correct_idx['clusters'], dtype=object)

order = np.argsort(weights)[::-1]
sorted_cluster_dict = {}
sorted_cluster_dict['clusters'] = clusters[order].tolist()
sorted_cluster_dict['weights']  = weights[order].tolist()

In [ ]:
weights = sorted_cluster_dict["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), weights)

plt.yticks([])
plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

weights = sorted_cluster_dict["weights"][:200]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70002)

plt.tight_layout()
plt.show()

medoids=[]
for i in range(len(sorted_cluster_dict["clusters"])):
    if sorted_cluster_dict["weights"][i] ==1 :
        medoids.append(sorted_cluster_dict["clusters"][i])
    else:
        dist_intra = dist_mat_1lvj[sorted_cluster_dict["clusters"][i].astype(int), :][:, sorted_cluster_dict["clusters"][i].astype(int)]
        medoids.append(sorted_cluster_dict["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

In [ ]:
clean_medoids = [
    m.item() if isinstance(m, np.ndarray) else int(m)
    for m in medoids
]

final_medoid_distances = dist_mat_1lvj[np.ix_(clean_medoids, clean_medoids)]

clusters = sorted_cluster_dict["clusters"]
weights  = sorted_cluster_dict["weights"]
dists    = final_medoid_distances

large_idxs = [i for i, c in enumerate(clusters) if c.size >= 10]
small_idxs = [i for i, c in enumerate(clusters) if c.size <  10]

In [ ]:
clusters_merged = []
weights_merged = []
for i in large_idxs:
    clusters_merged.append(clusters[i])
    weights_merged.append(weights[i])
for i in small_idxs:
    min_dist = np.min(dists[i, large_idxs])
    if min_dist < 0.8:
        closest_large_idx = large_idxs[np.argmin(dists[i, large_idxs])]
        clusters_merged[closest_large_idx] = np.concatenate((clusters_merged[closest_large_idx], clusters[i].reshape(-1)))
        weights_merged[closest_large_idx] += weights[i]
    else:
        clusters_merged.append(clusters[i])
        weights_merged.append(weights[i])

In [ ]:
clustering_dictionary_cleaned_singleton= {
    "clusters": clusters_merged,
    "weights": weights_merged
}

weights = clustering_dictionary_cleaned_singleton["weights"]

plt.figure(figsize=(10, 6))

plt.bar(range(len(weights)), np.array(weights)/70001)

plt.tight_layout()
plt.show()

w_row = np.asarray(weights_data[7])

cluster_weight_sums = [
    w_row[np.asarray(m, dtype=int)].sum()
    for m in clustering_dictionary_cleaned_singleton["clusters"]
]

plt.figure(figsize=(10, 6))

plt.bar(range(len(cluster_weight_sums)), cluster_weight_sums)

plt.xlabel("cluster index")
plt.tight_layout()
plt.show()

In [ ]:
medoids =[]
for i in range(len(clustering_dictionary_cleaned_singleton["clusters"])):
    if clustering_dictionary_cleaned_singleton["weights"][i] ==1 :
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i])
    else:
        dist_intra = dist_mat_1lvj[clustering_dictionary_cleaned_singleton["clusters"][i].astype(int), :][:, clustering_dictionary_cleaned_singleton["clusters"][i].astype(int)]
        medoids.append(clustering_dictionary_cleaned_singleton["clusters"][i][np.argmin(np.sum(dist_intra, axis=0))])

recap_dictionary={}
recap_dictionary["clusters"] = clustering_dictionary_cleaned_singleton["clusters"]
recap_dictionary["weights"]  = clustering_dictionary_cleaned_singleton["weights"]
recap_dictionary["medoids"]  = medoids
recap_dictionary["reweights"] = cluster_weight_sums

### Crystal-pocket distance


In [ ]:
gvecs_medoids = np.zeros((len(medoids), gvecs_dyn["1LVJ"][0].shape[1], gvecs_dyn["1LVJ"][0].shape[2], 4))
for n,i in enumerate(medoids):
    gvecs_medoids[n,:] = gvecs_dyn["1LVJ"][0][int(i)]

dist = cdist(gvecs_medoids.reshape(len(medoids), -1), results["1LVJ"].reshape(len(results["1LVJ"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

closest_centroid = np.argmin(dist, axis=1)

## Induced fit effect


In [ ]:
nearest_medoid={}
nearest_medoid["1QD3"]=3
nearest_medoid["1UTS"]=123
nearest_medoid["1UUD"]=61
nearest_medoid["1UUI"]=91
nearest_medoid["2L8H"]=24
nearest_medoid["1LVJ"]=307

In [ ]:
with open("data/recap_dictionary_clustering_1qd3.pkl", "rb") as f:
    recap_dictionary_clustering_1qd3 = pickle.load(f)

with open("data/recap_dictionary_clustering_1uts.pkl", "rb") as f:
    recap_dictionary_clustering_1uts = pickle.load(f)

with open("data/recap_dictionary_clustering_1uud.pkl", "rb") as f:
    recap_dictionary_clustering_1uud = pickle.load(f)

with open("data/recap_dictionary_clustering_1uui.pkl", "rb") as f:
    recap_dictionary_clustering_1uui = pickle.load(f)

with open("data/recap_dictionary_clustering_2l8h_correct.pkl", "rb") as f:
    recap_dictionary_clustering_2l8h = pickle.load(f)

with open("data/recap_dictionary_clustering_1lvj.pkl", "rb") as f:
    recap_dictionary_clustering_1lvj = pickle.load(f)

In [ ]:
peso_pre_1qd3=recap_dictionary_clustering_1qd3["weights"][nearest_medoid["1QD3"]]/70001
peso_post_1qd3=recap_dictionary_clustering_1qd3["reweights"][nearest_medoid["1QD3"]]

peso_pre_1uts=recap_dictionary_clustering_1uts["weights"][nearest_medoid["1UTS"]]/70001
peso_post_1uts=recap_dictionary_clustering_1uts["reweights"][nearest_medoid["1UTS"]]

peso_pre_1uud=recap_dictionary_clustering_1uud["weights"][nearest_medoid["1UUD"]]/70001
peso_post_1uud=recap_dictionary_clustering_1uud["reweights"][nearest_medoid["1UUD"]]

peso_pre_1uui=recap_dictionary_clustering_1uui["weights"][nearest_medoid["1UUI"]]/70001
peso_post_1uui=recap_dictionary_clustering_1uui["reweights"][nearest_medoid["1UUI"]]

peso_pre_2l8h=recap_dictionary_clustering_2l8h["weights"][nearest_medoid["2L8H"]]/70001
peso_post_2l8h=recap_dictionary_clustering_2l8h["reweights"][nearest_medoid["2L8H"]]

peso_pre_1lvj=recap_dictionary_clustering_1lvj["weights"][nearest_medoid["1LVJ"]]/70001
peso_post_1lvj=recap_dictionary_clustering_1lvj["reweights"][nearest_medoid["1LVJ"]]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1qd3["medoids"]]

medoids_1qd3 = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["1QD3"][0][medoids_1qd3]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1QD3"][0][medoids_1qd3]), -1), results["1QD3"].reshape(len(results["1QD3"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

In [ ]:
cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

pre_peso_07_1qd3=np.sum(recap_dictionary_clustering_1qd3["weights"][i] for i in cluster_07)/70001
pre_peso_08_1qd3=np.sum(recap_dictionary_clustering_1qd3["weights"][i] for i in cluster_08)/70001
pre_peso_09_1qd3=np.sum(recap_dictionary_clustering_1qd3["weights"][i] for i in cluster_09)/70001
pre_peso_10_1qd3=np.sum(recap_dictionary_clustering_1qd3["weights"][i] for i in cluster_10)/70001

post_peso_07_1qd3=np.sum(recap_dictionary_clustering_1qd3["reweights"][i] for i in cluster_07)
post_peso_08_1qd3=np.sum(recap_dictionary_clustering_1qd3["reweights"][i] for i in cluster_08)
post_peso_09_1qd3=np.sum(recap_dictionary_clustering_1qd3["reweights"][i] for i in cluster_09)
post_peso_10_1qd3=np.sum(recap_dictionary_clustering_1qd3["reweights"][i] for i in cluster_10)

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uts["medoids"]]

medoids_1uts = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["1UTS"][0][medoids_1uts]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UTS"][0][medoids_1uts]), -1), results["1UTS"].reshape(len(results["1UTS"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

In [ ]:
pre_peso_07_1uts=np.sum(recap_dictionary_clustering_1uts["weights"][i] for i in cluster_07)/70001
pre_peso_08_1uts=np.sum(recap_dictionary_clustering_1uts["weights"][i] for i in cluster_08)/70001
pre_peso_09_1uts=np.sum(recap_dictionary_clustering_1uts["weights"][i] for i in cluster_09)/70001
pre_peso_10_1uts=np.sum(recap_dictionary_clustering_1uts["weights"][i] for i in cluster_10)/70001

post_peso_07_1uts=np.sum(recap_dictionary_clustering_1uts["reweights"][i] for i in cluster_07)
post_peso_08_1uts=np.sum(recap_dictionary_clustering_1uts["reweights"][i] for i in cluster_08)
post_peso_09_1uts=np.sum(recap_dictionary_clustering_1uts["reweights"][i] for i in cluster_09)
post_peso_10_1uts=np.sum(recap_dictionary_clustering_1uts["reweights"][i] for i in cluster_10)

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uud["medoids"]]

medoids_1uud = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["1UUD"][0][medoids_1uud]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UUD"][0][medoids_1uud]), -1), results["1UUD"].reshape(len(results["1UUD"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

pre_peso_07_1uud=np.sum(recap_dictionary_clustering_1uud["weights"][i] for i in cluster_07)/70001
pre_peso_08_1uud=np.sum(recap_dictionary_clustering_1uud["weights"][i] for i in cluster_08)/70001
pre_peso_09_1uud=np.sum(recap_dictionary_clustering_1uud["weights"][i] for i in cluster_09)/70001
pre_peso_10_1uud=np.sum(recap_dictionary_clustering_1uud["weights"][i] for i in cluster_10)/70001

In [ ]:
post_peso_07_1uud=np.sum(recap_dictionary_clustering_1uud["reweights"][i] for i in cluster_07)
post_peso_08_1uud=np.sum(recap_dictionary_clustering_1uud["reweights"][i] for i in cluster_08)
post_peso_09_1uud=np.sum(recap_dictionary_clustering_1uud["reweights"][i] for i in cluster_09)
post_peso_10_1uud=np.sum(recap_dictionary_clustering_1uud["reweights"][i] for i in cluster_10)

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uui["medoids"]]

medoids_1uui = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["1UUI"][0][medoids_1uui]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UUI"][0][medoids_1uui]), -1), results["1UUI"].reshape(len(results["1UUI"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

pre_peso_07_1uui=np.sum(recap_dictionary_clustering_1uui["weights"][i] for i in cluster_07)/70001
pre_peso_08_1uui=np.sum(recap_dictionary_clustering_1uui["weights"][i] for i in cluster_08)/70001
pre_peso_09_1uui=np.sum(recap_dictionary_clustering_1uui["weights"][i] for i in cluster_09)/70001
pre_peso_10_1uui=np.sum(recap_dictionary_clustering_1uui["weights"][i] for i in cluster_10)/70001

post_peso_07_1uui=np.sum(recap_dictionary_clustering_1uui["reweights"][i] for i in cluster_07)
post_peso_08_1uui=np.sum(recap_dictionary_clustering_1uui["reweights"][i] for i in cluster_08)
post_peso_09_1uui=np.sum(recap_dictionary_clustering_1uui["reweights"][i] for i in cluster_09)
post_peso_10_1uui=np.sum(recap_dictionary_clustering_1uui["reweights"][i] for i in cluster_10)

In [ ]:
cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_2l8h["medoids"]]

medoids_2l8h = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["2L8H"][0][medoids_2l8h]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["2L8H"][0][medoids_2l8h]), -1), results["2L8H"].reshape(len(results["2L8H"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

pre_peso_07_2l8h=np.sum(recap_dictionary_clustering_2l8h["weights"][i] for i in cluster_07)/70001
pre_peso_08_2l8h=np.sum(recap_dictionary_clustering_2l8h["weights"][i] for i in cluster_08)/70001
pre_peso_09_2l8h=np.sum(recap_dictionary_clustering_2l8h["weights"][i] for i in cluster_09)/70001
pre_peso_10_2l8h=np.sum(recap_dictionary_clustering_2l8h["weights"][i] for i in cluster_10)/70001

post_peso_07_2l8h=np.sum(recap_dictionary_clustering_2l8h["reweights"][i] for i in cluster_07)
post_peso_08_2l8h=np.sum(recap_dictionary_clustering_2l8h["reweights"][i] for i in cluster_08)
post_peso_09_2l8h=np.sum(recap_dictionary_clustering_2l8h["reweights"][i] for i in cluster_09)
post_peso_10_2l8h=np.sum(recap_dictionary_clustering_2l8h["reweights"][i] for i in cluster_10)

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1lvj["medoids"]]

medoids_1lvj = np.array(cleaned_medoids, dtype=int)

gvecs_medoids=gvecs_dyn["1LVJ"][0][medoids_1lvj]

In [ ]:
dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1LVJ"][0][medoids_1lvj]), -1), results["1LVJ"].reshape(len(results["1LVJ"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

cluster_07=np.where(dist[0,:]<0.7)[0]
cluster_08=np.where(dist[0,:]<0.8)[0]
cluster_09=np.where(dist[0,:]<0.9)[0]
cluster_10=np.where(dist[0,:]<1.0)[0]

pre_peso_07_1lvj=np.sum(recap_dictionary_clustering_1lvj["weights"][i] for i in cluster_07)/70001
pre_peso_08_1lvj=np.sum(recap_dictionary_clustering_1lvj["weights"][i] for i in cluster_08)/70001
pre_peso_09_1lvj=np.sum(recap_dictionary_clustering_1lvj["weights"][i] for i in cluster_09)/70001
pre_peso_10_1lvj=np.sum(recap_dictionary_clustering_1lvj["weights"][i] for i in cluster_10)/70001

post_peso_07_1lvj=np.sum(recap_dictionary_clustering_1lvj["reweights"][i] for i in cluster_07)
post_peso_08_1lvj=np.sum(recap_dictionary_clustering_1lvj["reweights"][i] for i in cluster_08)
post_peso_09_1lvj=np.sum(recap_dictionary_clustering_1lvj["reweights"][i] for i in cluster_09)
post_peso_10_1lvj=np.sum(recap_dictionary_clustering_1lvj["reweights"][i] for i in cluster_10)

In [ ]:
mpl.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "TeX Gyre Termes", "Nimbus Roman No9 L", "DejaVu Serif"],
    "axes.labelsize": 15,
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "grid.color":    "grey",
    "grid.linestyle":"--",
    "grid.alpha":    0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

def _uniformize(ax, twin=None):
    ax.set_axisbelow(True)
    ax.grid(True, linestyle="--")
    ax.tick_params(axis='both', labelsize=14)

    if twin is not None:
        twin.tick_params(axis='y', labelsize=14)

In [ ]:
labels = ["1QD3", "1UTS", "1UUD", "1UUI", "2L8H", "1LVJ"]
x = np.arange(len(labels))
width = 0.35

datasets = [
    {
        "title": "cutoff 0.8",
        "pre": [pre_peso_08_1qd3, pre_peso_08_1uts, pre_peso_08_1uud, pre_peso_08_1uui, pre_peso_08_2l8h, pre_peso_08_1lvj],
        "post": [post_peso_08_1qd3, post_peso_08_1uts, post_peso_08_1uud, post_peso_08_1uui, post_peso_08_2l8h, post_peso_08_1lvj],
    },
    {
        "title": "cutoff 0.9",
        "pre": [pre_peso_09_1qd3, pre_peso_09_1uts, pre_peso_09_1uud, pre_peso_09_1uui, pre_peso_09_2l8h, pre_peso_09_1lvj],
        "post": [post_peso_09_1qd3, post_peso_09_1uts, post_peso_09_1uud, post_peso_09_1uui, post_peso_09_2l8h, post_peso_09_1lvj],
    },
    {
        "title": "cutoff 1.0",
        "pre": [pre_peso_10_1qd3, pre_peso_10_1uts, pre_peso_10_1uud, pre_peso_10_1uui, pre_peso_10_2l8h, pre_peso_10_1lvj],
        "post": [post_peso_10_1qd3, post_peso_10_1uts, post_peso_10_1uud, post_peso_10_1uui, post_peso_10_2l8h, post_peso_10_1lvj],
    },
]

before_color = "#20A39E"
after_color  = "#FFBA49"

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True, constrained_layout=True)

for ax, ds in zip(axes, datasets):
    bars1 = ax.bar(x - width/2, ds["pre"], width, label="Before Reweighting", color=before_color, alpha=0.9)
    bars2 = ax.bar(x + width/2, ds["post"], width, label="After Reweighting",  color=after_color,  alpha=0.95)

    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=0)
    ax.set_ylim(0, 0.3)

    ax.grid(axis="y", linestyle=":", linewidth=0.8, alpha=0.6, zorder=0)
    for spine in ["top", "right"]:
        ax.spines[spine].set_visible(False)

    ax.set_xlabel(ds["title"])

axes[0].set_ylabel("Weight")

for ax in axes:
    _uniformize(ax)

handles, leg_labels = axes[0].get_legend_handles_labels()
fig.legend(handles, leg_labels, loc="upper center", ncol=2, frameon=True,
           bbox_to_anchor=(0.5, 1.07), borderaxespad=0.0, fontsize=15)
plt.savefig("images_paper/comparison_weights_before_after_reweighting.png", dpi=600,bbox_inches='tight')
plt.show()

In [ ]:
threshold=5

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_1qd3["reweights"])

free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

medoid_indices=np.where(free_energies<threshold)[0]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1qd3["medoids"]]

medoids_1qd3 = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["1QD3"][0][medoids_1qd3]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1QD3"][0][medoids_1qd3]), -1), results["1QD3"].reshape(len(results["1QD3"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

idx_tbs_1qd3=np.argmin(dist[0,:])

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_1uts["reweights"])

In [ ]:
free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

medoid_indices=np.where(free_energies<threshold)[0]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uts["medoids"]]

medoids_1uts = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["1UTS"][0][medoids_1uts]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UTS"][0][medoids_1uts]), -1), results["1UTS"].reshape(len(results["1UTS"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

idx_tbs_1uts=np.argmin(dist[0,:])

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_1uud["reweights"])

free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

medoid_indices=np.where(free_energies<threshold)[0]

In [ ]:
cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uud["medoids"]]

medoids_1uud = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["1UUD"][0][medoids_1uud]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UUD"][0][medoids_1uud]), -1), results["1UUD"].reshape(len(results["1UUD"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

idx_tbs_1uud=np.argmin(dist[0,:])

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_1uui["reweights"])

free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

medoid_indices=np.where(free_energies<5)[0]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1uui["medoids"]]

medoids_1uui = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["1UUI"][0][medoids_1uui]

In [ ]:
dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1UUI"][0][medoids_1uui]), -1), results["1UUI"].reshape(len(results["1UUI"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

idx_tbs_1uui=np.argmin(dist[0,:])

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_2l8h["reweights"])

free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

plt.plot(free_energies)

medoid_indices=np.where(free_energies<5)[0]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_2l8h["medoids"]]

medoids_2l8h = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["2L8H"][0][medoids_2l8h]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["2L8H"][0][medoids_2l8h]), -1), results["2L8H"].reshape(len(results["2L8H"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

In [ ]:
idx_tbs_2l8h=np.argmin(dist[0,:])

R = 0.001987
T = 300.0

reweights = np.array(recap_dictionary_clustering_1lvj["reweights"])

free_energies = -R * T * np.log(reweights)

free_energies -= free_energies.min()

medoid_indices=np.where(free_energies<5)[0]

cleaned_medoids = [int(np.array(x).item()) for x in recap_dictionary_clustering_1lvj["medoids"]]

medoids_1lvj = np.array(cleaned_medoids, dtype=int)[medoid_indices]

gvecs_medoids=gvecs_dyn["1LVJ"][0][medoids_1lvj]

dist = cdist(gvecs_medoids.reshape(len(gvecs_dyn["1LVJ"][0][medoids_1lvj]), -1), results["1LVJ"].reshape(len(results["1LVJ"]),-1), metric='euclidean')
dist = dist / np.sqrt(gvecs_medoids.shape[1])

dist=dist.T

idx_tbs_1lvj=np.argmin(dist[0,:])

In [ ]:
idx_tbs_1lvj_frame=int(recap_dictionary_clustering_1lvj["medoids"][idx_tbs_1lvj])
idx_tbs_2l8h_frame=int(recap_dictionary_clustering_2l8h["medoids"][idx_tbs_2l8h])
idx_tbs_1uui_frame=int(recap_dictionary_clustering_1uui["medoids"][idx_tbs_1uui])
idx_tbs_1uud_frame=int(recap_dictionary_clustering_1uud["medoids"][idx_tbs_1uud])
idx_tbs_1uts_frame=int(recap_dictionary_clustering_1uts["medoids"][idx_tbs_1uts])
idx_tbs_1qd3_frame=int(recap_dictionary_clustering_1qd3["medoids"][idx_tbs_1qd3])



In [ ]:
BINS         = 100
SMOOTH_SIGMA = 3
INTERP       = "none"
PCTL_RANGE   = (0, 100)
MARKER_SIZE  = 85

TEMP_K = 300.0
K_B_KCAL_PER_MOLK = 0.0019872041
KBT = K_B_KCAL_PER_MOLK * TEMP_K

EPS          = 1e-7

GAMMA        = 0.9
CAP_PCTL     = 98.5



In [ ]:
def make_soft_color_contour():
    colors = [
        (0.25, 0.42, 0.70),
        (0.55, 0.70, 0.88),
        (0.75, 0.85, 0.90),
        (1.00, 1.00, 1.00),
    ]
    return LinearSegmentedColormap.from_list("soft_color_contour", colors)

In [ ]:
def fes_from_points(x, y, w=None, bins=BINS, smooth_sigma=SMOOTH_SIGMA, kbt=KBT):
    xlo, xhi = np.percentile(x, PCTL_RANGE); ylo, yhi = np.percentile(y, PCTL_RANGE)
    H, xe, ye = np.histogram2d(
        x, y, bins=bins, range=[[xlo, xhi], [ylo, yhi]],
        weights=None if w is None else w, density=True
    )
    if smooth_sigma and smooth_sigma > 0:
        try:
            H = gaussian_filter(H, sigma=smooth_sigma, mode="nearest")
        except Exception:
            pass
    F = -kbt * np.log(H + EPS)
    F -= np.nanmin(F)
    return xe, ye, F

def plot_fes_contour(dyn_coords, weights, crystal_coords, pdb):
    x, y = dyn_coords[:, 0], dyn_coords[:, 1]
    w = None if weights is None else np.asarray(weights, dtype=float)
    if w is not None:
        w = np.clip(w, 0.0, None)
        w = w / w.sum()

    xe, ye, F = fes_from_points(x, y, w)
    extent = [xe[0], xe[-1], ye[0], ye[-1]]

    X, Y = np.meshgrid((xe[:-1]+xe[1:])/2, (ye[:-1]+ye[1:])/2)

    vmax = np.nanpercentile(F, CAP_PCTL)
    norm = Normalize(vmin=0, vmax=vmax)

    fig, ax = plt.subplots(figsize=(5.8,5.8), dpi=600)

    n_levels = 10
    cf = ax.contourf(X, Y, F.T, levels=n_levels,
                 cmap=make_soft_color_contour(),
                 norm=norm, alpha=0.95, zorder=1)

    cs = ax.contour(X, Y, F.T, levels=n_levels,
                colors="k", linewidths=0.4, alpha=0.35, zorder=2)

    ax.contourf(
        X, Y, F.T,
        levels=[vmax, F.max()+1],
        colors=[(1,1,1,1)]
    )
    cbar = fig.colorbar(cf, ax=ax, pad=0.015, fraction=0.046)
    cbar.set_label(r"$\Delta$F (kcal/mol)", rotation=270, labelpad=14, fontsize=20)
    cbar.ax.tick_params(labelsize=20, width=0.5, length=2)
    cbar.outline.set_linewidth(0.5)

    fc = color_for(pdb)
    ax.scatter(crystal_coords[:,0], crystal_coords[:,1],
               s=MARKER_SIZE*HALO_SCALE, facecolor="white", edgecolor="none", zorder=3)
    ax.scatter(crystal_coords[:,0], crystal_coords[:,1],
               s=MARKER_SIZE, facecolor=fc, edgecolor="black", label=pdb, linewidth=0.7, zorder=4)
    if pdb in medoid_idx:
        m = medoid_idx[pdb]
        if 0 <= m < dyn_coords.shape[0]:
            sx, sy = dyn_coords[m, 0], dyn_coords[m, 1]
            ax.scatter([sx], [sy], s=STAR_SIZE*1.25, marker="*", facecolor="white",
                       edgecolor="white", linewidth=0.0, zorder=6)
            ax.scatter([sx], [sy], s=STAR_SIZE, marker="*", facecolor=STAR_COLOR,
                       edgecolor="black", linewidth=0.9, zorder=7, label="closest centroid")

    ax.set_xlabel(f"PC1 ", fontsize=20)
    ax.set_ylabel(f"PC2 ", fontsize=20)
    sns.despine(ax=ax)

    _uniformize(ax)
    ax.set_axisbelow(True)
    ax.grid(True, linestyle="--")
    ax.tick_params(axis='both', labelsize=20)

    leg = ax.legend(frameon=True, fontsize=18, loc="lower right")
    leg.get_frame().set_edgecolor("white")
    leg.get_frame().set_linewidth(1.8)
    leg.get_frame().set_facecolor((1, 1, 1, 0.0))

    plt.tight_layout()
    return fig

In [ ]:
MARKER_SIZE  = 85
HALO_SCALE   = 1.35
STAR_SIZE    = 140
STAR_COLOR   = "#7F00FF"

POINT_COLORS = [
    "#EF476F",
    "#F78C6B",
    "#FFD166",
    "#06D6A0",
    "#118AB2",
    "#073B4C"
]

pdb_order = [
    "1LVJ","1QD3","1UTS","1UUD","1UUI","2L8H"
]
pdb_point_color = {pdb: POINT_COLORS[i % len(POINT_COLORS)] for i, pdb in enumerate(pdb_order)}
def color_for(pdb: str) -> str:
    return pdb_point_color.get(pdb, POINT_COLORS[hash(pdb) % len(POINT_COLORS)])

debug=0
for pdb in pockets.keys():
    if pdb in EXCLUDE_GVEC or pdb not in gvecs_pockets_dyn:
        continue

    crystal = results[pdb]
    dyn     = gvecs_pockets_dyn[pdb][0]

    dyn_flat     = dyn.reshape(len(dyn), -1)
    crystal_flat = crystal.reshape(len(crystal), -1)

    pca         = PCA(n_components=2)
    dyn_pca     = pca.fit_transform(dyn_flat)
    crystal_pca = pca.transform(crystal_flat)

    w = np.asarray(weights_data[7], dtype=float).ravel()
    if w.shape[0] != len(dyn_pca):
        w = w[:len(dyn_pca)]

    fig = plot_fes_contour(dyn_pca, weights=w, crystal_coords=crystal_pca, pdb=pdb)
    plt.savefig(f"./images_paper/fes_{pdb}_blue.png", dpi=600, bbox_inches='tight')

    plt.show()

## Druggability


SiteMap scores are aligned to the trajectory frames so missing frames can be retained with neutral placeholder values.


In [ ]:
data=pd.read_csv("data/sitescores_mae.txt", sep="\t")

list_idx=np.arange(70001)

present = pd.to_numeric(data["frame"], errors="coerce").dropna().astype(int).unique()
missing = np.setdiff1d(np.arange(70001), present)

missing_data = pd.DataFrame(missing, columns=["frame"])
missing_data["SiteScore1"]=0
missing_data["SiteScore2"]=0
missing_data["SiteScore3"]=0
missing_data["SiteScore4"]=0

concat_data = pd.concat([data, missing_data], ignore_index=True)

concat_data.sort_values(by="frame", inplace=True)
concat_data.reset_index(drop=True, inplace=True)

## SHAPE

In [ ]:
data_shape_md=np.empty((70002,29))
c2 = trj.select_atoms('name C2 and resid 1:29')
n_res = len(c2)               
n_pairs = n_res - 1          
n_frames = trj.trajectory.n_frames

In [ ]:
all_pos = np.array([ c2.positions.copy() for ts in trj.trajectory ])

In [ ]:
diff = all_pos[:, 1:, :] - all_pos[:, :-1, :]
distances = np.linalg.norm(diff, axis=2)

In [ ]:
def weighted_std(values, weights, axis=0, ddof=0):
    w = np.asarray(weights, dtype=float)
    avg = np.average(values, axis=axis, weights=w)
    var = np.average((values - avg)**2, axis=axis, weights=w)

    if ddof:
        n_eff = w.sum()**2 / (w**2).sum()
        var *= n_eff / (n_eff - ddof)

    return np.sqrt(var)

In [ ]:
fluct_C2C2= distances.std(axis=0)

In [ ]:
data_shape_exp=pd.read_csv("data/Dataset_S1.csv")

In [ ]:
data_red=data_shape_exp.loc[16:40]

In [ ]:
rmsf_orig_w = weighted_std(
    values   = distances,
    weights  = w_full,  
    axis     = 0,
    ddof     = 1
)

In [ ]:
n_res    = all_pos.shape[1]
n_frames = all_pos.shape[0]

diff        = all_pos[:, 1:, :] - all_pos[:, :-1, :]
distances   = np.linalg.norm(diff, axis=2)                 
rmsf_orig   = distances.std(axis=0)                        



In [ ]:
def internucleotide_distance_fluctuations(all_pos, weights=None):
    n_frames, n_res, _ = all_pos.shape

    if weights is None:
        weights = np.ones(n_frames)

    d_pairs = np.linalg.norm(
        all_pos[:, 1:, :] - all_pos[:, :-1, :],
        axis=-1
    )  

    fluc_pairs = weighted_std(d_pairs, weights, axis=0)

    metric = np.empty(n_res)
    metric[0] = fluc_pairs[0]
    metric[-1] = fluc_pairs[-1]
    metric[1:-1] = 0.5 * (fluc_pairs[:-1] + fluc_pairs[1:])

    return metric


In [ ]:
shape_pre = internucleotide_distance_fluctuations(all_pos)
shape_post = internucleotide_distance_fluctuations(all_pos, weights=w_full)